<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: a model can outperform a hand-written rule when several safe signals are combined. My methodology question is: Was the comparison made on data the model did not see, and was the same split used for both the rule and the model? A model-vs-rule result is only meaningful when both are evaluated on the same held-out population and metric.

Finding 2: search/content signals can be useful for prioritization. My methodology question is: Were those signals available before the outcome, or do they contain the answer in disguise? In this project, trend_direction and trend_pct are excluded because the target is defined from them. Feature importance is treated as an association in this dataset, not proof that a signal causes a decline.

These questions are constructive: they do not reject the findings; they identify the validation and feature-timing evidence needed for the claims to carry weight

In [1]:
from pathlib import Path
import os, urllib.request
DATA_NAME = "content_refresh_anonymized.csv"
HERE = Path.cwd().resolve()
data_path = next((p / "data" / "raw" / DATA_NAME for p in [HERE, *HERE.parents] if (p / "data" / "raw" / DATA_NAME).exists()), None)
if data_path is None:
    ROOT = HERE / "flyrank_notebook_workspace"
    data_path = ROOT / "data" / "raw" / DATA_NAME
    data_path.parent.mkdir(parents=True, exist_ok=True)
    if not data_path.exists():
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv", data_path)
        except Exception as exc:
            raise FileNotFoundError("Extract the internship ZIP and open this notebook from it, or provide the public starter CSV.") from exc
else:
    ROOT = data_path.parents[2]
os.chdir(ROOT)
print("Dataset:", data_path)
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv(data_path)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
for raw in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    df[f"log_{raw}"] = np.log1p(pd.to_numeric(df[raw], errors="coerce").clip(lower=0))

numeric = [c for c in ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "days_with_impressions", "content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "scroll_rate", "word_count"] if c in df]
categorical = [c for c in ["content_type", "main_intent", "freshness_tier", "position_tier"] if c in df]
X = df[numeric + categorical]
y = df["is_declining_label"]
groups = df["client_id"].astype(str)

def make_pipeline(include_leaky_feature=False):
    safe_prep = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])
    if include_leaky_feature:
        prep = ColumnTransformer([("safe", safe_prep, numeric + categorical), ("leaky_trend_pct", SimpleImputer(strategy="median"), ["trend_pct"])])
    else:
        prep = safe_prep
    return Pipeline([("prep", prep), ("model", RandomForestClassifier(n_estimators=250, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=42, n_jobs=-1))])

def precision_at_50(y_true, scores):
    order = np.argsort(-np.asarray(scores))[:50]
    return float(np.asarray(y_true)[order].mean())

print("Target definition: trend_direction == 'down'")
print("Rows:", len(df), "| Clients:", groups.nunique(), "| Positive-label rate:", round(y.mean(), 3))
print("Legal feature groups:", numeric, categorical)


Dataset: /content/flyrank_notebook_workspace/data/raw/content_refresh_anonymized.csv
Target definition: trend_direction == 'down'
Rows: 30000 | Clients: 32 | Positive-label rate: 0.542
Legal feature groups: ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'days_with_impressions', 'content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count'] ['content_type', 'main_intent', 'freshness_tier', 'position_tier']


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The “before” score below is a conventional random row split. It can place pages from one client in both train and test, which may make the task easier than real deployment.

The “after” score is the honest version: a client holdout. All pages from the test clients are unseen during training. I report the difference rather than selecting the more flattering number. The model is judged by Precision@50 because the intended use is to prioritize the first 50 pages for human review.

In [2]:
# BEFORE: random rows can mix a client across train and test.
train_random, test_random = train_test_split(np.arange(len(df)), test_size=0.20, stratify=y, random_state=42)
random_model = make_pipeline().fit(X.iloc[train_random], y.iloc[train_random])
random_scores = random_model.predict_proba(X.iloc[test_random])[:, 1]

# AFTER: client holdout keeps each client entirely in train or test.
train_grouped, test_grouped = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(X, y, groups=groups))
grouped_model = make_pipeline().fit(X.iloc[train_grouped], y.iloc[train_grouped])
grouped_scores = grouped_model.predict_proba(X.iloc[test_grouped])[:, 1]

comparison = pd.DataFrame([
    {"split": "Random row split (before)", "ROC-AUC": roc_auc_score(y.iloc[test_random], random_scores), "Average precision": average_precision_score(y.iloc[test_random], random_scores), "Precision@50": precision_at_50(y.iloc[test_random], random_scores), "base rate": y.iloc[test_random].mean()},
    {"split": "Client holdout (after)", "ROC-AUC": roc_auc_score(y.iloc[test_grouped], grouped_scores), "Average precision": average_precision_score(y.iloc[test_grouped], grouped_scores), "Precision@50": precision_at_50(y.iloc[test_grouped], grouped_scores), "base rate": y.iloc[test_grouped].mean()},
])
print(comparison.round(3).to_string(index=False))
comparison.to_csv("w06_honest_split_comparison.csv", index=False)


                    split  ROC-AUC  Average precision  Precision@50  base rate
Random row split (before)    0.757              0.771          0.90      0.542
   Client holdout (after)    0.628              0.617          0.68      0.511


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Timeline: all legal feature values are the trailing-window visibility, freshness, position, CTR, engagement, and content signals supplied at the point of scoring. The label is the dataset's observed trend_direction == "down" category.

Label-derived columns excluded: trend_direction and trend_pct. trend_pct is the numeric change underlying the label category, so it is the answer in disguise.
Identifiers excluded: content_id and client_id. Client ID is used only to form the grouped split.

Decision-derived columns excluded: no product flags or existing action scores are used as inputs. Such a field could be a baseline but not a feature.

The test below deliberately includes trend_pct once. A suspicious jump is the expected sign that the audit is working. The final model must use the non-leaky client-holdout result.

In [3]:
# Attack test: trend_pct is deliberately added only to demonstrate label leakage.
leaky_frame = df[numeric + categorical + ["trend_pct"]]
leaky_model = make_pipeline(include_leaky_feature=True).fit(leaky_frame.iloc[train_grouped], y.iloc[train_grouped])
leaky_scores = leaky_model.predict_proba(leaky_frame.iloc[test_grouped])[:, 1]

leakage_check = pd.DataFrame([
    {"feature set": "Final safe features", "ROC-AUC": roc_auc_score(y.iloc[test_grouped], grouped_scores), "Precision@50": precision_at_50(y.iloc[test_grouped], grouped_scores)},
    {"feature set": "Unsafe test: final features + trend_pct", "ROC-AUC": roc_auc_score(y.iloc[test_grouped], leaky_scores), "Precision@50": precision_at_50(y.iloc[test_grouped], leaky_scores)},
])
print(leakage_check.round(3).to_string(index=False))
print("Final model exclusion list: trend_direction, trend_pct, content_id, client_id, and any product/action flags.")
leakage_check.to_csv("w06_leakage_audit.csv", index=False)



                            feature set  ROC-AUC  Precision@50
                    Final safe features    0.628          0.68
Unsafe test: final features + trend_pct    0.999          1.00
Final model exclusion list: trend_direction, trend_pct, content_id, client_id, and any product/action flags.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest original sentence: “The model predicts which content refreshes will improve Google rankings.”

Rewritten claim: “On a client-holdout split of the anonymized starter release, the model ranked pages in the observed declining-trend category at the reported Precision@50. The resulting queue is a directional decision-support tool for human review. It does not show that refreshing a page causes a change in visibility, and it does not model Google's algorithm.”

In [4]:
# Save a small, public-safe receipt of this audit.
audit_receipt = {
    "random_row_precision_at_50": float(comparison.loc[0, "Precision@50"]),
    "client_holdout_precision_at_50": float(comparison.loc[1, "Precision@50"]),
    "safe_feature_auc": float(leakage_check.loc[0, "ROC-AUC"]),
    "leaky_feature_auc": float(leakage_check.loc[1, "ROC-AUC"]),
    "final_claim": "Client-holdout ranking of the observed declining-trend category; decision support only.",
}
import json
with open("w06_validation_receipt.json", "w", encoding="utf-8") as f:
    json.dump(audit_receipt, f, indent=2)
print("Wrote w06_honest_split_comparison.csv, w06_leakage_audit.csv, and w06_validation_receipt.json")



Wrote w06_honest_split_comparison.csv, w06_leakage_audit.csv, and w06_validation_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.